## 0. Importing necessary libraries,<br> Creating a folder for storing python scripts,<br> Setting up device-agnostic code

Since we're going to be creating Python scripts out of our most useful code cells, let's create a folder for storing those scripts.

We'll call the folder `TinyVGG_pytorch` and create it using Python's [`os.makedirs()`](https://docs.python.org/3/library/os.html) method.

In [28]:
import os
import torch
from torch import nn

torch.__version__

'2.6.0+cu124'

Creating a folder for python scripts

In [2]:
# os.makedirs("going_modular", exist_ok=True)

Setting up device-agnostic code

In [29]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

# temp = torch.cuda.device_count()
# temp
# torch.cuda.get_device_capability()


# import psutil
# psutil.cpu_count(logical=False)

'cuda'

## 1. Get data


Data might be downloaded as splitted (ready-made) or not. If not, we need to take care of it by splitting by ourself own <br>
If the image folder doesn't exist, download it as "Retinal_image_all folder" and prepare it... 

### (If necessary) Preparing the dataset into class-folders using csv info - maybe splitting isn't necessary when we have "train.csv" and "test.csv"

In [4]:
from pathlib import Path
import shutil
from tqdm import tqdm

source_dir = Path("/kaggle/input/ham10000-sp/Images splitted/Splitted")
output_dir = Path("/kaggle/working/all_images")
output_dir.mkdir(exist_ok=True)

for split in ["train", "test"]:
    split_dir = source_dir / split
    for class_dir in split_dir.iterdir():
        target_class_dir = output_dir / class_dir.name
        target_class_dir.mkdir(exist_ok=True)

        for img in tqdm(list(class_dir.iterdir()), desc=f"Merging {class_dir.name}"):
            shutil.copy(img, target_class_dir / img.name)


Merging bcc: 100%|██████████| 102/102 [00:01<00:00, 95.81it/s]


In [5]:
from pathlib import Path
from torchvision import datasets, transforms
from torchvision.transforms.functional import InterpolationMode

IMG_SIZE = 450

# Train augmentation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomPerspective(distortion_scale=0.2),
    transforms.Normalize([0.5018, 0.5015, 0.5013],
                         [0.1029, 0.0985, 0.0807])
])

# Validation/test transform
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5018, 0.5015, 0.5013],
                         [0.1029, 0.0985, 0.0807])
])

# Path to your combined dataset
data_path = Path("/kaggle/working/all_images")

# Create the full dataset WITHOUT transform
full_dataset = datasets.ImageFolder(root=data_path, transform=None)


In [6]:
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset, DataLoader
import numpy as np

targets = np.array(full_dataset.targets)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

BATCH_SIZE = 16
fold_loaders = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(targets)), targets)):
    print(f"Preparing Fold {fold+1}...")

    # Subset datasets
    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)

    # Apply transforms AFTER splitting
    train_subset.dataset.transform = train_transform
    val_subset.dataset.transform = val_transform

    # Create DataLoaders
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=4, pin_memory=True)

    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=4, pin_memory=True)

    fold_loaders.append((train_loader, val_loader))

print("\n✔ 5-Fold DataLoaders ready!")


Preparing Fold 1...
Preparing Fold 2...
Preparing Fold 3...
Preparing Fold 4...
Preparing Fold 5...

✔ 5-Fold DataLoaders ready!


In [7]:
current_fold = 3  # choose 0-4

train_dataloader, test_dataloader = fold_loaders[current_fold]
print(f"Checking Fold {current_fold+1}")


Checking Fold 4


In [31]:
import gc
torch.cuda.empty_cache()
gc.collect()

0

In [8]:
print(f"Fold {current_fold+1} sizes:")
print("Train samples:", len(train_dataloader.dataset))
print("Test samples:", len(test_dataloader.dataset))
# train_loader.dataset is a Subset
train_targets = np.array([train_dataloader.dataset[i][1] for i in range(len(train_dataloader.dataset))])
test_targets   = np.array([test_dataloader.dataset[i][1] for i in range(len(test_dataloader.dataset))])
class_names = full_dataset.classes

train_dist = {class_names[i]: np.sum(train_targets == i) for i in range(len(class_names))}
test_dist   = {class_names[i]: np.sum(test_targets == i) for i in range(len(class_names))}

print(f"Fold {current_fold+1} Train Distribution:\n", train_dist)
print(f"Fold {current_fold+1} Test Distribution:\n", test_dist)

Fold 4 sizes:
Train samples: 8012
Test samples: 2003
Fold 4 Train Distribution:
 {'akiec': 262, 'bcc': 411, 'bkl': 879, 'df': 92, 'mel': 891, 'nv': 5364, 'vasc': 113}
Fold 4 Test Distribution:
 {'akiec': 65, 'bcc': 103, 'bkl': 220, 'df': 23, 'mel': 222, 'nv': 1341, 'vasc': 29}


# Up

In [32]:
import torchvision

# Get a set of pretrained model weights
weights = torchvision.models.ResNeXt101_64X4D_Weights.DEFAULT # .DEFAULT = best available weights from pretraining on ImageNet
weights

ResNeXt101_64X4D_Weights.IMAGENET1K_V1

In [33]:
# Get the transforms used to create our pretrained weights
auto_transforms = weights.transforms()
auto_transforms

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [11]:
# Check out single image size/shape
img, label = next(iter(train_dataloader))

# Batch size will now be 1, try changing the batch_size parameter above and see what happens
print(f"Image shape: {img.shape} -> [batch_size, color_channels, height, width]")
print(f"Label shape: {label.shape}")

Image shape: torch.Size([16, 3, 450, 450]) -> [batch_size, color_channels, height, width]
Label shape: torch.Size([16])


### Experimentations

In [12]:
# Get a batch of images
image_batch, label_batch = next(iter(train_dataloader))

# Get a single image from the batch
image, label = image_batch[0], label_batch[0]

# View the batch shapes
image.shape, label

(torch.Size([3, 450, 450]), tensor(5))

### Getting pretrained EfficientNetb5

In [34]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential as Seq
# from gcn_lib import Grapher, act_layer

# for Kaggle import
import sys
sys.path.append("/kaggle/input/vig-pytorch/")
# from gcn_lib import Grapher, act_layer


from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from timm.models.helpers import load_pretrained
from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from timm.models.registry import register_model

In [35]:
model = torchvision.models.resnext101_64x4d(weights=weights).to(device)

In [36]:
from torchinfo import summary

# Create an instance of TransformerEncoderBlock
model_summary = model

# Print an input and output summary of our Transformer Encoder (uncomment for full output)
summary(model=model_summary,
        input_size=(16, 3, 224, 224), # (batch_size, num_patches, embedding_dimension)
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                  Input Shape          Output Shape         Param #              Trainable
ResNet (ResNet)                          [16, 3, 224, 224]    [16, 1000]           --                   True
├─Conv2d (conv1)                         [16, 3, 224, 224]    [16, 64, 112, 112]   9,408                True
├─BatchNorm2d (bn1)                      [16, 64, 112, 112]   [16, 64, 112, 112]   128                  True
├─ReLU (relu)                            [16, 64, 112, 112]   [16, 64, 112, 112]   --                   --
├─MaxPool2d (maxpool)                    [16, 64, 112, 112]   [16, 64, 56, 56]     --                   --
├─Sequential (layer1)                    [16, 64, 56, 56]     [16, 256, 56, 56]    --                   True
│    └─Bottleneck (0)                    [16, 64, 56, 56]     [16, 256, 56, 56]    --                   True
│    │    └─Conv2d (conv1)               [16, 64, 56, 56]     [16, 256, 56, 56]    16,384               True
│    │    └─BatchN

### Freezing the base model and changing output layer 

In [40]:
# # Freeze all base layers in the "features" section of the model (the feature extractor) by setting requires_grad=False
# for param in model.features.parameters():
#     param.requires_grad = False

In [37]:
# Set the manual seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Get the length of class_names (one output unit for each class)
output_shape = len(class_names)

# Recreate the classifier layer and seed it to the target device
model.fc = torch.nn.Linear(in_features=2048, 
                    out_features=output_shape, # same number of output units as our number of classes
                    bias=True).to(device)

In [38]:
# Print an input and output summary of our Transformer Encoder (uncomment for full output)
summary(model=model_summary,
        input_size=(16, 3, 224, 224), # (batch_size, num_patches, embedding_dimension)
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                  Input Shape          Output Shape         Param #              Trainable
ResNet (ResNet)                          [16, 3, 224, 224]    [16, 7]              --                   True
├─Conv2d (conv1)                         [16, 3, 224, 224]    [16, 64, 112, 112]   9,408                True
├─BatchNorm2d (bn1)                      [16, 64, 112, 112]   [16, 64, 112, 112]   128                  True
├─ReLU (relu)                            [16, 64, 112, 112]   [16, 64, 112, 112]   --                   --
├─MaxPool2d (maxpool)                    [16, 64, 112, 112]   [16, 64, 56, 56]     --                   --
├─Sequential (layer1)                    [16, 64, 56, 56]     [16, 256, 56, 56]    --                   True
│    └─Bottleneck (0)                    [16, 64, 56, 56]     [16, 256, 56, 56]    --                   True
│    │    └─Conv2d (conv1)               [16, 64, 56, 56]     [16, 256, 56, 56]    16,384               True
│    │    └─BatchN

In [39]:
model = nn.DataParallel(model)  ### for two GPU faster computations 

In [44]:
# model.load_state_dict(torch.load('/kaggle/input/efficientnetb5-trained-models/efficientnetb5_0.8016304347826086_acc - wandb - EfficientNetB5  Dataset - Aptos Gaussian Filtered EHB Tunnings - SAMPLER Batch32 Final Experimentations  Selected.pth'))

In [45]:
# import wandb

# # start a new wandb run to track this script
# tracking = wandb.init(
#     # set the wandb project where this run will be logged
#     project="Glaucoma Fundus Imaging",
#     name="HAM10000: ResNext101",
# #     name="Origa: Fb_model",
#     notes="",
#     # track hyperparameters and run metadata
#     config={
#         "learning_rate": 2e-3,
#         "architecture": "fb",
#         "dataset": "Default",
#         "epochs": 200,
#     }
# )

# # 4b7dfb240ea0d5aae1afac2de518c0e940547396

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

  ········································


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


# **Training**

In [40]:
"""
Contains functions for training and testing a PyTorch model.
"""
from typing import Dict, List, Tuple
from sklearn.metrics import classification_report

import torch

from tqdm.auto import tqdm
import wandb


def train_step(model: torch.nn.Module, 
               dataloader: torch.utils.data.DataLoader, 
               loss_fn: torch.nn.Module, 
               optimizer: torch.optim.Optimizer,
               device: torch.device,
               len_train_data) -> Tuple[float, float]:
    """Trains a PyTorch model for a single epoch.

    Turns a target PyTorch model to training mode and then
    runs through all of the required training steps (forward
    pass, loss calculation, optimizer step).

    Args:
    model: A PyTorch model to be trained.
    dataloader: A DataLoader instance for the model to be trained on.
    loss_fn: A PyTorch loss function to minimize.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
    A tuple of training loss and training accuracy metrics.
    In the form (train_loss, train_accuracy). For example:

    (0.1112, 0.8743)
    """
    # Put model in train mode
    model.train()

    # Setup train loss and train accuracy values
    train_loss = 0.0 
    train_acc = 0.0

    # for evaluation
    y_true_train_data = []
    y_predicted_train_data = []

    # Loop through data loader data batches
    for batch, (X, y) in enumerate(dataloader):
        # Send data to target device
        X, y = X.to(device), y.to(device)


        # 1. Forward pass
        y_pred = model(X)
        
#         y_pred = y_pred.logits

        # 2. Calculate  and accumulate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item() 

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # Calculate and accumulate accuracy metric across all batches
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        # train_acc += (y_pred_class == y).sum().item()/len(y_pred)
        train_acc += (y_pred_class == y).sum().item()
        
        # for evaluaiton
        y_true_train_data.append(y)
        y_predicted_train_data.append(y_pred_class)

    # Adjust metrics to get average loss and accuracy per batch 
    train_loss = train_loss / len(dataloader)
#     train_acc = train_acc / len(dataloader)
    train_acc = train_acc / len_train_data
    
    #     New way of accuracy calculation
    true = 0
    tot = 0
    for i in range(len(y_true_train_data)):
        x = (y_true_train_data[i] == y_predicted_train_data[i])
        for j in range(len(x)):
            tot += 1
            if x[j] == True:
                true += 1
    train_acc = (true/tot)

    # return train_loss, train_acc
    return train_loss, train_acc, y_true_train_data, y_predicted_train_data




def test_step(model: torch.nn.Module, 
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device,
              len_test_data) -> Tuple[float, float]:
    """Tests a PyTorch model for a single epoch.

    Turns a target PyTorch model to "eval" mode and then performs
    a forward pass on a testing dataset.

    Args:
    model: A PyTorch model to be tested.
    dataloader: A DataLoader instance for the model to be tested on.
    loss_fn: A PyTorch loss function to calculate loss on the test data.
    device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
    A tuple of testing loss and testing accuracy metrics.
    In the form (test_loss, test_accuracy). For example:

    (0.0223, 0.8985)
    """
    # Put model in eval mode
    model.eval() 

    # Setup test loss and test accuracy values
    test_loss = 0.0 
    test_acc = 0.0

    # for evaluation
    y_true_test_data = []
    y_predicted_test_data = []


    # Turn on inference context manager
    with torch.inference_mode():

        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_pred_logits = model(X)
            
#             test_pred_logits = test_pred_logits.logits
            
            # 2. Calculate and accumulate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # Calculate and accumulate accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            # test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))
            test_acc += (test_pred_labels == y).sum().item()

            # test_pred_labels = torch.argmax(torch.softmax(test_pred_logits, dim=1), dim=1)
            # test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels)) 
            

            # for evaluaiton
            y_true_test_data.append(y)
            y_predicted_test_data.append(test_pred_labels)


    # Adjust metrics to get average loss and accuracy per batch 
    test_loss = test_loss / len(dataloader)
#     test_acc = test_acc / len(dataloader)
    test_acc = test_acc / len_test_data


    
#     New way of accuracy calculation
    true = 0
    tot = 0
    for i in range(len(y_true_test_data)):
        x = (y_true_test_data[i] == y_predicted_test_data[i])
        for j in range(len(x)):
            tot += 1
            if x[j] == True:
                true += 1
    test_acc = (true/tot)
    
    # return test_loss, test_acc
    return test_loss, test_acc, y_true_test_data, y_predicted_test_data




def train(model: torch.nn.Module, 
          train_dataloader: torch.utils.data.DataLoader, 
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device,
          # tracking,
          best_accuracy,
          len_train_data,
          len_test_data,
        ) -> Dict[str, List[float]]:
    """Trains and tests a PyTorch model.

    Passes a target PyTorch models through train_step() and test_step()
    functions for a number of epochs, training and testing the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Args:
    model: A PyTorch model to be trained and tested.
    train_dataloader: A DataLoader instance for the model to be trained on.
    test_dataloader: A DataLoader instance for the model to be tested on.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    loss_fn: A PyTorch loss function to calculate loss on both datasets.
    epochs: An integer indicating how many epochs to train for.
    device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
    A dictionary of training and testing loss as well as training and
    testing accuracy metrics. Each metric has a value in a list for 
    each epoch.
    In the form: {train_loss: [...],
              train_acc: [...],
              test_loss: [...],
              test_acc: [...]} 
    For example if training for epochs=2: 
             {train_loss: [2.0616, 1.0537],
              train_acc: [0.3945, 0.3945],
              test_loss: [1.2641, 1.5706],
              test_acc: [0.3400, 0.2973]} 
    """
    # Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
    }


        
    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc, y_true_train_data, y_predicted_train_data = train_step(model=model,
                                                                                      dataloader=train_dataloader,
                                                                                      loss_fn=loss_fn,
                                                                                      optimizer=optimizer,
                                                                                      device=device,
                                                                                      len_train_data=len_train_data,
                                                                                      )
        test_loss, test_acc, y_true_test_data, y_predicted_test_data = test_step(model=model,
                                                                                 dataloader=test_dataloader,
                                                                                 loss_fn=loss_fn,
                                                                                 device=device,
                                                                                 len_test_data=len_test_data,
                                                                                 )

        if best_accuracy < test_acc:
            print(f"\nAccuracy Improved ({best_accuracy} to {test_acc}), Saving the model...............................................\n")
            best_accuracy = test_acc
            best_y_true_test_data = y_true_test_data
            best_y_predicted_test_data = y_predicted_test_data

            model_name_path = f"{best_accuracy}_fold_{current_fold+1}_acc.pth"
            torch.save(model.state_dict(), model_name_path)
            
            
                        
            # taking "y_predicted_train_data" & "y_true_train_data" into 1D array because it came out as batch by batch 2D list
            predicted_train_data_1D = []
            true_train_data_1D = []

            for i in range(len(y_predicted_train_data)):
                for j in range(len(y_predicted_train_data[i])):
                    predicted_train_data_1D.append(y_predicted_train_data[i][j])
                    true_train_data_1D.append(y_true_train_data[i][j])

            # taking both into CPU
            predicted_train_data_cpu = torch.tensor(predicted_train_data_1D, device = 'cpu')
            true_train_data_cpu = torch.tensor(true_train_data_1D, device = 'cpu')



            # now same procedure for test data's
            predicted_test_data_1D = []
            true_test_data_1D = []

            for i in range(len(y_predicted_test_data)):
                for j in range(len(y_predicted_test_data[i])):
                    predicted_test_data_1D.append(y_predicted_test_data[i][j])
                    true_test_data_1D.append(y_true_test_data[i][j])

            # taking both into CPU
            predicted_test_data_cpu = torch.tensor(predicted_test_data_1D, device = 'cpu')
            true_test_data_cpu = torch.tensor(true_test_data_1D, device = 'cpu')



            # now same procedure for best test data's
            best_predicted_test_data_1D = []
            best_true_test_data_1D = []

            for i in range(len(best_y_predicted_test_data)):
                for j in range(len(best_y_predicted_test_data[i])):
                    best_predicted_test_data_1D.append(best_y_predicted_test_data[i][j])
                    best_true_test_data_1D.append(best_y_true_test_data[i][j])

            # taking both into CPU
            best_predicted_test_data_cpu = torch.tensor(best_predicted_test_data_1D, device = 'cpu')
            best_true_test_data_cpu = torch.tensor(best_true_test_data_1D, device = 'cpu')
            
            

            # Generate a classification report
            # F1 score on train data
            y_true_train = true_train_data_cpu
            y_pred_train = predicted_train_data_cpu

            report = classification_report(y_true_train, y_pred_train, target_names=class_names)

            print(f"Evaluation report on Train data: \n\n{report}\n\n\n\n")


            # F1 score on test data
            y_true_test = true_test_data_cpu
            y_pred_test = predicted_test_data_cpu

            report = classification_report(y_true_test, y_pred_test, target_names=class_names)

            print(f"Evaluation report on Test data: \n\n{report}")
            
            
            
            
            
            
        #     Live result tracking#####################################
        #     for i in range(len(y_true_test_data)):
        #         print(f"{y_true_test_data[i]} and {y_predicted_test_data[i]}")
        acc_0 = 0
        acc_1 = 0
        total_0 = 0
        total_1 = 0
        for i in range(len(y_predicted_test_data)):
            for j in range(len(y_true_test_data[i])):
                if (y_true_test_data[i][j] == 0):
                    total_0 += 1
                    if (y_true_test_data[i][j] == y_predicted_test_data[i][j]):
                        acc_0 += 1
                elif (y_true_test_data[i][j] == 1): 
                    total_1 += 1
                    if (y_true_test_data[i][j] == y_predicted_test_data[i][j]):
                        acc_1 += 1
#         print(f"\n0: {acc_0}/{total_0} and 1: {acc_1}/{total_1}\n\n")
        #     Live result tracking#####################################

        
        # Print out what's happening
        print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f} | "
#           f"0: {acc_0}/{total_0} | "
#           f"1: {acc_1}/{total_1}"
        )
        

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        
        # tracking.log({
        #     "train_acc": train_acc,
        #     "test_acc": test_acc,
        # })

    # Return the filled results at the end of the epochs
    # return results
    

    return results, y_true_train_data, y_predicted_train_data, y_true_test_data, y_predicted_test_data, best_accuracy, best_y_true_test_data, best_y_predicted_test_data


In [41]:
# from going_modular import engine

# Setup the optimizer to optimize our ViT model parameters using hyperparameters from the ViT paper 
optimizer = torch.optim.AdamW(params=model.parameters(), 
#                               lr=2e-3, # Base LR from Table 3 for ViT-* ImageNet-1k                                    ## learning rate decre..
                              lr = 1e-3,
                              betas=(0.9, 0.999), # default values but also mentioned in ViT paper section 4.1 (Training & Fine-tuning)
                              weight_decay=0.05,
#                               weight_decay=1e-5
                              ) # from the ViT paper section 4.1 (Training & Fine-tuning) and Table 3 for ViT-* ImageNet-1k

# lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=5)
# Setup the loss function for multi-class classification
loss_fn = torch.nn.CrossEntropyLoss()

# Set the seeds
# set_seeds()
torch.manual_seed(42)
torch.cuda.manual_seed(42)

best_accuracy=0.0

In [ ]:
# Train the model and save the training results to a dictionary
results, y_true_train_data, y_predicted_train_data, y_true_test_data, y_predicted_test_data, best_accuracy, best_y_true_test_data, best_y_predicted_test_data = train(model=model,
                                                                                                           train_dataloader=train_dataloader,
                                                                                                           test_dataloader=test_dataloader,
                                                                                                           optimizer=optimizer,
                                                                                                           loss_fn=loss_fn,
                                                                                                           epochs=30,
                                                                                                           device=device,
                                                                                                           # tracking=tracking,
                                                                                                           best_accuracy=best_accuracy,
                                                                                                           len_train_data=len(train_dataloader.dataset),
                                                                                                           len_test_data=len(test_dataloader.dataset)
                                                                                                           )

  0%|          | 0/30 [00:00<?, ?it/s]


Accuracy Improved (0.0 to 0.7209186220668996), Saving the model...............................................

Evaluation report on Train data: 

              precision    recall  f1-score   support

       akiec       0.33      0.13      0.19       262
         bcc       0.36      0.22      0.27       411
         bkl       0.41      0.30      0.35       879
          df       0.00      0.00      0.00        92
         mel       0.41      0.20      0.27       891
          nv       0.77      0.94      0.84      5364
        vasc       0.53      0.23      0.32       113

    accuracy                           0.70      8012
   macro avg       0.40      0.29      0.32      8012
weighted avg       0.64      0.70      0.66      8012





Evaluation report on Test data: 

              precision    recall  f1-score   support

       akiec       1.00      0.02      0.03        65
         bcc       0.40      0.49      0.44       103
         bkl       0.38      0.51      0.44       220


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Accuracy Improved (0.7209186220668996 to 0.7498751872191712), Saving the model...............................................

Evaluation report on Train data: 

              precision    recall  f1-score   support

       akiec       0.44      0.24      0.31       262
         bcc       0.48      0.44      0.46       411
         bkl       0.52      0.44      0.48       879
          df       0.00      0.00      0.00        92
         mel       0.49      0.27      0.35       891
          nv       0.81      0.94      0.87      5364
        vasc       0.75      0.58      0.65       113

    accuracy                           0.74      8012
   macro avg       0.50      0.41      0.44      8012
weighted avg       0.71      0.74      0.72      8012





Evaluation report on Test data: 

              precision    recall  f1-score   support

       akiec       0.37      0.38      0.38        65
         bcc       0.33      0.86      0.48       103
         bkl       0.80      0.17      

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classif


Accuracy Improved (0.7498751872191712 to 0.7683474787818273), Saving the model...............................................

Evaluation report on Train data: 

              precision    recall  f1-score   support

       akiec       0.51      0.37      0.43       262
         bcc       0.60      0.60      0.60       411
         bkl       0.57      0.51      0.54       879
          df       0.00      0.00      0.00        92
         mel       0.55      0.34      0.42       891
          nv       0.84      0.94      0.89      5364
        vasc       0.82      0.64      0.72       113

    accuracy                           0.78      8012
   macro avg       0.56      0.48      0.51      8012
weighted avg       0.75      0.78      0.76      8012





Evaluation report on Test data: 

              precision    recall  f1-score   support

       akiec       0.88      0.11      0.19        65
         bcc       0.39      0.88      0.54       103
         bkl       0.78      0.40      

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch: 4 | train_loss: 0.5680 | train_acc: 0.7986 | test_loss: 0.6951 | test_acc: 0.7499 | 

Accuracy Improved (0.7683474787818273 to 0.7713429855217174), Saving the model...............................................

Evaluation report on Train data: 

              precision    recall  f1-score   support

       akiec       0.63      0.53      0.58       262
         bcc       0.68      0.72      0.70       411
         bkl       0.66      0.62      0.64       879
          df       0.47      0.08      0.13        92
         mel       0.62      0.42      0.50       891
          nv       0.87      0.95      0.91      5364
        vasc       0.84      0.76      0.80       113

    accuracy                           0.81      8012
   macro avg       0.68      0.58      0.61      8012
weighted avg       0.80      0.81      0.80      8012





Evaluation report on Test data: 

              precision    recall  f1-score   support

       akiec       0.51      0.65      0.57        65
 

In [132]:
import torch
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [26]:
import gc
torch.cuda.empty_cache()
gc.collect()


0

### Final Result for Publication

In [ ]:
# from typing import Dict, List, Tuple
# import torch
# from tqdm.auto import tqdm

# def test_step(model: torch.nn.Module, 
#               dataloader: torch.utils.data.DataLoader,
#               loss_fn: torch.nn.Module,
#               device: torch.device) -> Tuple[float, float]:

#     # Put model in eval mode
#     model.eval() 
#     # Setup test loss and test accuracy values
#     test_loss, test_acc = 0, 0
#     # for evaluation
#     y_true_test_data = []
#     y_predicted_test_data = []
    
#     # Turn on inference context manager
#     with torch.inference_mode():
#         # torch.manual_seed(42)
#         # Loop through DataLoader batches
#         for batch, (X, y) in enumerate(dataloader):

#             # Send data to target device
#             X, y = X.to(device), y.to(device)
#             # 1. Forward pass
#             test_pred_logits = model(X)
#             # 2. Calculate and accumulate loss
#             loss = loss_fn(test_pred_logits, y)
#             test_loss += loss.item()

#             # Calculate and accumulate accuracy
#             test_pred_labels = test_pred_logits.argmax(dim=1)
#             test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))

#             # test_pred_labels = torch.argmax(torch.softmax(test_pred_logits, dim=1), dim=1)
#             # test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels)) 

#             # for evaluaiton
#             y_true_test_data.append(y)
#             y_predicted_test_data.append(test_pred_labels)


#     # Adjust metrics to get average loss and accuracy per batch 
#     test_loss = test_loss / len(dataloader)
#     test_acc = test_acc / len(dataloader)

#     # return test_loss, test_acc
#     return test_loss, test_acc, y_true_test_data, y_predicted_test_data



# epochs = 1
# for epoch in tqdm(range(epochs)):
#         test_loss, test_acc, y_true_test_data, y_predicted_test_data = test_step(model=model,
#                                                                                  dataloader=test_dataloader,
#                                                                                  loss_fn=loss_fn,
#                                                                                  device=device)
#         best_accuracy = test_acc
#         best_y_true_test_data = y_true_test_data
#         best_y_predicted_test_data = y_predicted_test_data


#         # Print out what's happening

#         print(
#           f"Epoch: {epoch+1} | "
#           f"test_loss: {test_loss:.4f} | "
#           f"test_acc: {test_acc:.4f}"
#         )


In [ ]:
# taking "y_predicted_train_data" & "y_true_train_data" into 1D array because it came out as batch by batch 2D list
predicted_train_data_1D = []
true_train_data_1D = []

for i in range(len(y_predicted_train_data)):
    for j in range(len(y_predicted_train_data[i])):
        predicted_train_data_1D.append(y_predicted_train_data[i][j])
        true_train_data_1D.append(y_true_train_data[i][j])

In [ ]:
Truee = 0
for i in range(len(true_train_data_1D)):
    if true_train_data_1D[i] == predicted_train_data_1D[i]:
        Truee += 1

print(Truee/len(true_train_data_1D))

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

### Evaluations

In [ ]:
# taking "y_predicted_train_data" & "y_true_train_data" into 1D array because it came out as batch by batch 2D list
predicted_train_data_1D = []
true_train_data_1D = []

for i in range(len(y_predicted_train_data)):
    for j in range(len(y_predicted_train_data[i])):
        predicted_train_data_1D.append(y_predicted_train_data[i][j])
        true_train_data_1D.append(y_true_train_data[i][j])

# taking both into CPU
predicted_train_data_cpu = torch.tensor(predicted_train_data_1D, device = 'cpu')
true_train_data_cpu = torch.tensor(true_train_data_1D, device = 'cpu')



# now same procedure for test data's
predicted_test_data_1D = []
true_test_data_1D = []

for i in range(len(y_predicted_test_data)):
    for j in range(len(y_predicted_test_data[i])):
        predicted_test_data_1D.append(y_predicted_test_data[i][j])
        true_test_data_1D.append(y_true_test_data[i][j])

# taking both into CPU
predicted_test_data_cpu = torch.tensor(predicted_test_data_1D, device = 'cpu')
true_test_data_cpu = torch.tensor(true_test_data_1D, device = 'cpu')



# now same procedure for best test data's
best_predicted_test_data_1D = []
best_true_test_data_1D = []

for i in range(len(best_y_predicted_test_data)):
    for j in range(len(best_y_predicted_test_data[i])):
        best_predicted_test_data_1D.append(best_y_predicted_test_data[i][j])
        best_true_test_data_1D.append(best_y_true_test_data[i][j])

# taking both into CPU
best_predicted_test_data_cpu = torch.tensor(best_predicted_test_data_1D, device = 'cpu')
best_true_test_data_cpu = torch.tensor(best_true_test_data_1D, device = 'cpu')

### Precision, Recall and F1 scores

In [ ]:
from sklearn.metrics import classification_report

# Generate a classification report


# F1 score on train data
y_true_train = true_train_data_cpu
y_pred_train = predicted_train_data_cpu

report = classification_report(y_true_train, y_pred_train, target_names=class_names)

print(f"Evaluation report on Train data: \n\n{report}\n\n\n\n")


# F1 score on test data
y_true_test = true_test_data_cpu
y_pred_test = predicted_test_data_cpu

report = classification_report(y_true_test, y_pred_test, target_names=class_names)

print(f"Evaluation report on Test data: \n\n{report}")


# F1 score on best test score
best_y_true_test = best_true_test_data_cpu
best_y_pred_test = best_predicted_test_data_cpu

report = classification_report(best_y_true_test, best_y_pred_test, target_names=class_names)

print(f"Evaluation report on Best Test score: \n\n{report}")

### Kappa score

In [ ]:
from torchmetrics.classification import MulticlassCohenKappa

metric = MulticlassCohenKappa(num_classes=5)

target = y_true_train
preds = y_pred_train
print(f"Kappa score on train data : {metric(preds, target)}")

target = y_true_test
preds = y_pred_test
print(f"Kappa score on test data : {metric(preds, target)}")


target = best_y_true_test
preds = best_y_pred_test
print(f"Kappa score on Best test score : {metric(preds, target)}")

### Train Confusion Matrix

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

# 2. Setup confusion matrix instance and compare predictions to targets
confmat = ConfusionMatrix(num_classes=len(class_names), task='multiclass')
confmat_tensor = confmat(preds=y_pred_train,
                         target=y_true_train)

# 3. Plot the confusion matrix
fig, ax = plot_confusion_matrix(
    conf_mat=confmat_tensor.numpy(), # matplotlib likes working with NumPy 
    class_names=class_names, # turn the row and column labels into class names
    figsize=(10, 7)
);

### Test Confusion Matrix

In [ ]:
# 2. Setup confusion matrix instance and compare predictions to targets
confmat = ConfusionMatrix(num_classes=len(class_names), task='multiclass')
confmat_tensor = confmat(preds=y_pred_test,
                         target=y_true_test)

# 3. Plot the confusion matrix
fig, ax = plot_confusion_matrix(
    conf_mat=confmat_tensor.numpy(), # matplotlib likes working with NumPy 
    class_names=class_names, # turn the row and column labels into class names
    figsize=(10, 7)
);

### Best Score Confusion Matrix

In [ ]:
# 2. Setup confusion matrix instance and compare predictions to targets
confmat = ConfusionMatrix(num_classes=len(class_names), task='multiclass')
confmat_tensor = confmat(preds=best_y_pred_test,
                         target=best_y_true_test)

# 3. Plot the confusion matrix
fig, ax = plot_confusion_matrix(
    conf_mat=confmat_tensor.numpy(), # matplotlib likes working with NumPy 
    class_names=class_names, # turn the row and column labels into class names
    figsize=(10, 7)
);

### Plottings

In [ ]:
# Get the plot_loss_curves() function from helper_functions.py, download the file if we don't have it
try:
    from going_modular.helper_functions import plot_loss_curves
except:
    print("[INFO] Couldn't find helper_functions.py, downloading...")
    # with open("helper_functions.py", "wb") as f:
    #     import requests
    #     request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
    #     f.write(request.content)
    # from helper_functions import plot_loss_curves

# Plot the loss curves of our model
plot_loss_curves(results)